# Checkpoint 2 — LED vs. laser optotagging
Four plots for discussion with Liz and Austin. These are **observational comparisons of recording periods**, not a validation of either hardware type. Hardware labels use the inherited session-ID cutoff (`789848216`) and need independent verification.

**Data and analysis setup**
Reuse the saved per-unit optotagging results and QC/candidate flags. Response rates refer to the established **8-ms evoked window**, not necessarily the entire illumination period. Reconnect `WD_BLACK` before running.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

ROOT = Path(os.environ.get("SPIKES2SLOPE_DATA", "/Volumes/WD_BLACK/Spikes2Slope"))
SOURCE = ROOT / "opto_results" / "all_opto_unit_results.csv"
if not SOURCE.is_file():
    raise FileNotFoundError(f"Connect WD_BLACK or set SPIKES2SLOPE_DATA: {SOURCE}")
df = pd.read_csv(SOURCE)
required = {"unit_id", "ecephys_session_id", "cre_line", "evoked_rate_hz",
            "baseline_rate_hz", "n_pulses", "matched_candidate", "snr_qc"}
assert required.issubset(df.columns), f"Missing columns: {required - set(df.columns)}"
assert df.unit_id.is_unique, "Expected one row per unit"
for flag in ["matched_candidate", "snr_qc"]:
    df[flag] = df[flag].astype(str).str.lower().map(
        {"true": True, "false": False, "1": True, "0": False})
    assert df[flag].notna().all(), f"Unexpected values in {flag}"
df["hardware"] = np.where(df.ecephys_session_id < 789848216, "LED", "Laser")
order = ["LED", "Laser"]
colors = {"LED": "#0072B2", "Laser": "#D55E00"}
sns.set_theme(style="whitegrid", context="notebook")
eligible = df.loc[df.snr_qc].copy()
candidates = df.loc[df.matched_candidate].copy()
assert (candidates.snr_qc).all()
session = df.groupby(["ecephys_session_id", "hardware", "cre_line"], as_index=False).agg(
    eligible=("snr_qc", "sum"), candidates=("matched_candidate", "sum"),
    n_pulses=("n_pulses", "first"))
session["yield_pct"] = np.where(session.eligible > 0,
                                100 * session.candidates / session.eligible, np.nan)
means = candidates.groupby("ecephys_session_id").evoked_rate_hz.mean().rename("mean_evoked_hz")
session = session.merge(means, on="ecephys_session_id", how="left")
assert session.groupby("ecephys_session_id").size().eq(1).all()
print(f"{len(session)} sessions, {len(eligible):,} QC-eligible units, {len(candidates):,} candidates.")
print("Hardware assignment: inherited session-period proxy; not independently verified.")

**Candidate counts by hardware**
LED has **694 candidates across 15 sessions**, versus **433 across 13 laser sessions**. The more meaningful comparison is yield among QC-eligible units: **16.2% LED versus 10.5% laser**. This is a descriptive difference, not proof of superior tagging.

In [ ]:
summary = session.groupby("hardware").agg(
    sessions=("ecephys_session_id", "size"),
    eligible=("eligible", "sum"), candidates=("candidates", "sum")).reindex(order)
summary["yield_pct"] = 100 * summary.candidates / summary.eligible
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8))
for ax, col, ylabel in zip(axes, ["candidates", "yield_pct"],
                            ["Candidate units", "Candidates / QC-eligible units (%)"]):
    vals = summary[col].to_numpy()
    ax.bar(order, vals, color=[colors[h] for h in order], width=.54)
    for i, v in enumerate(vals):
        ax.text(i, v, f"{v:.0f}" if col=="candidates" else f"{v:.1f}%",
                ha="center", va="bottom", fontsize=11, weight="bold")
    ax.set(ylabel=ylabel); ax.set_ylim(0, max(vals)*1.23)
axes[0].set_title("Absolute candidate count")
axes[1].set_title("Candidate yield")
for i,h in enumerate(order):
    axes[0].text(i, -max(summary.candidates)*.13, f"n={summary.loc[h,'sessions']} sessions",
                 ha="center", fontsize=9)
fig.tight_layout()
plt.show()

**Evoked firing-rate distributions**
Among identified candidates, the processed data show **higher mean evoked rates for laser (52.12 Hz) than LED (11.18 Hz)**—opposite the meeting-note direction. The distributions reveal whether this reflects most candidates or a strong high-rate tail; rates are not themselves measures of optotagging reliability.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
for h in order:
    x = np.sort(candidates.loc[candidates.hardware.eq(h), "evoked_rate_hz"].dropna().to_numpy())
    y = np.arange(1, len(x)+1) / len(x)
    ax.step(x, y, where="post", linewidth=2, color=colors[h],
            label=f"{h}: n={len(x)}, median={np.median(x):.1f} Hz")
ax.set(xlabel="Evoked firing rate (Hz, 8-ms response window)",
       ylabel="Fraction of candidates at or below rate",
       title="Candidate evoked firing-rate distributions", ylim=(0, 1.01))
ax.legend(frameon=False)
sns.despine()
fig.tight_layout()
plt.show()

**Firing rates vary across sessions**
The session-level Welch comparison of candidate mean rates was **LED − laser = −32.83 Hz (95% CI −47.00 to −18.67; p ≈ 0.00025)** in the original run. Show each session rather than treating 1,127 units as independent experiments. Statistical significance does not establish hardware causation.

In [ ]:
s = session.dropna(subset=["mean_evoked_hz"]).copy()
a = s.loc[s.hardware.eq("LED"), "mean_evoked_hz"].to_numpy()
b = s.loc[s.hardware.eq("Laser"), "mean_evoked_hz"].to_numpy()
test = stats.ttest_ind(a, b, equal_var=False)
fig, ax = plt.subplots(figsize=(7.5, 4.3))
sns.stripplot(data=s, x="hardware", y="mean_evoked_hz", hue="cre_line",
              order=order, jitter=.15, size=9, alpha=.88, ax=ax)
for i, h in enumerate(order):
    sub=s.loc[s.hardware.eq(h), "mean_evoked_hz"]
    ax.hlines(sub.mean(), i-.23, i+.23, color="black", linewidth=2.5)
ax.set(xlabel="", ylabel="Session mean evoked rate of candidates (Hz)",
       title=f"Session-level response strength (Welch p={test.pvalue:.3g})")
ax.legend(title="Targeted Cre line", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
sns.despine()
fig.tight_layout()
plt.show()
print(f"Sessions: LED={len(a)}, laser={len(b)}; mean difference LED - laser={a.mean()-b.mean():.2f} Hz.")
print("Interpret cautiously: hardware coincides with recording era, stimulation protocol, and session composition.")

**Austin's plot — fraction of weakly responsive candidates**
Use **mean evoked spikes per pulse** rather than *exactly one total spike*, because LED has 45 selected pulses and laser generally has 75 or 151. The original **37.6% versus 0%** exact-total-spike comparison is mechanically biased by exposure and the candidate threshold. Here **≤0.1 spike/pulse** is a clearly labeled exploratory cutoff, not a validated tagging criterion; session points show its variability.

In [ ]:
WINDOW_S = .008
candidates["spikes_per_pulse"] = candidates.evoked_rate_hz * WINDOW_S
THRESH = .10  # exploratory, not a validated optotagging threshold
candidates["weak"] = candidates.spikes_per_pulse <= THRESH
weak = candidates.groupby("hardware").agg(n=("unit_id","size"), weak=("weak","sum")).reindex(order)
weak["pct"] = 100 * weak.weak / weak.n
per_session = candidates.groupby(["ecephys_session_id","hardware"]).weak.agg(
    n="size", fraction="mean").reset_index()
fig, axes = plt.subplots(1, 2, figsize=(10, 3.9), gridspec_kw={"width_ratios":[1,1.25]})
axes[0].bar(order, weak.pct, color=[colors[h] for h in order], width=.55)
for i,h in enumerate(order):
    row=weak.loc[h]
    axes[0].text(i, row.pct + 1.5, f"{row.pct:.1f}% ({int(row.weak)}/{int(row.n)})",
                 ha="center", fontsize=10)
axes[0].set(ylabel="Weak candidates (%)", ylim=(0, max(100, weak.pct.max()*1.22)),
            title="Pooled candidate percentages")
sns.stripplot(data=per_session.assign(pct=100*per_session.fraction),
              x="hardware", y="pct", order=order, hue="hardware", palette=colors,
              jitter=.17, size=7, alpha=.85, ax=axes[1], legend=False)
axes[1].set(xlabel="", ylabel="Weak candidates per session (%)",
            title="Each dot = one session")
fig.suptitle("Weak response: ≤0.1 mean evoked spike per pulse (exploratory)", fontsize=12)
fig.tight_layout()
plt.show()
print(weak[["weak", "n", "pct"]].round(1))
print("Cannot infer trial-by-trial response reliability from aggregate spike counts.")

**Interpretation for Liz and Austin**
The available data support a **hardware-period association** in candidate yield and evoked rates. They do **not yet justify dropping LED or laser candidates**. Before exclusion, independently verify hardware assignment and compare pulse-resolved response reliability, latency, artifacts, calibrated illumination, and genotype/session composition. Even the revised weak-response cutoff remains exploratory.